# Overnight sweep — twelve runs, unattended

Start this before you sleep, with **Save Version → Save and Run All** so it keeps
going after the browser closes. Expect 5–7 hours.

## What it produces

| Stage | Runs | Purpose |
|---|---|---|
| gate | 1 × 2 epochs | proves the recipe learns before anything long starts |
| sweep | 4 architectures × 3 seeds × 12 epochs | the comparison that Stage B needs |
| extra seeds | 2 more EfficientNet-B0 seeds | tighter variance on the reference model, only if time allows |

Everything lands in one zip that the last cell writes **even if earlier stages
failed**.

## Why you can leave it running

Five things could waste a night. Each one is handled:

| Failure | Handling |
|---|---|
| The recipe still cannot learn | the gate stops it in ~5 minutes; the sweep never starts |
| One run crashes (OOM, NaN, bug) | caught per run, recorded in `_overnight_log.json`, the loop continues |
| One run hangs | subprocess timeout kills it and moves on |
| The session dies halfway | finished seeds are skipped on the next run, so a restart resumes |
| The session hits its time limit | a budget stops launching new runs ~45 minutes before the limit |

## The recipe, and why it is not negotiable

`protocol_v2` restores the settings that produced the 99.54% reference: AdamW
lr 1e-4, weight decay 1e-4, global batch 32, constant learning rate, no label
smoothing, no class weights, **random shuffling**.

That last one matters. v1 and the first v2 gate used blocked shuffling, and the
manifest is written in class order, so batches contained a median of 3 classes
instead of 12. That broke BatchNorm: the only architecture that trained was
ConvNeXt, which uses LayerNorm and ignores batch composition. Evidence in
`docs/KNOWN_ISSUES.md` R14.

**Do not "improve" the batch size or the learning rate in this notebook.** The
one time that was done, twelve runs and 5.6 GPU-hours produced nothing usable.

## 1. Setup and preflight

Everything that can be checked in ten seconds is checked here: two GPUs present, dataset attached, manifests agree, images resolve, disk space for the cache.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch, torchvision

# ---- knobs ------------------------------------------------------------------
SESSION_BUDGET_H = 10.0    # stop starting new runs after this many hours
RESERVE_H = 0.75           # leave this much at the end for results + download
ALLOW_EXTRA_SEEDS = True   # 2 extra EfficientNet-B0 seeds if the budget allows
GATE_EPOCHS = 2
GATE_THRESHOLD = 0.95

STARTED_AT = time.time()

# ---- inputs -----------------------------------------------------------------
DATASET_ROOT = Path("/kaggle/input/datasets/shubhamkumar12864/plantvillage")
KAGGLE_INPUT = Path("/kaggle/input")


def autodiscover_dataset_root(preferred: Path) -> Path:
    if (preferred / "data" / "manifests" / "plantvillage_development.csv").is_file():
        return preferred
    if KAGGLE_INPUT.is_dir():
        for manifests in sorted(KAGGLE_INPUT.rglob("manifests")):
            if (manifests / "plantvillage_development.csv").is_file():
                return manifests.parent.parent
    return preferred


_resolved = autodiscover_dataset_root(DATASET_ROOT)
if _resolved != DATASET_ROOT:
    print(f"dataset root corrected: {DATASET_ROOT} -> {_resolved}")
DATASET_ROOT = _resolved

MANIFEST_DIR = DATASET_ROOT / "data" / "manifests"
IMAGE_ROOT = DATASET_ROOT / "data" / "raw" / "PlantVillage"
TRAIN_MANIFEST = MANIFEST_DIR / "plantvillage_development.csv"

WORKDIR = Path("/kaggle/working")
SCRIPT = WORKDIR / "train_ddp.py"
WORKER = WORKDIR / "sweep_worker.py"
PROTOCOL_PATH = WORKDIR / "protocol_v2.json"
CACHE_DIR = Path("/kaggle/temp/plantvillage_cache")
OUTPUT_ROOT = WORKDIR / "runs"
GATE_ROOT = OUTPUT_ROOT / "_gate"
LOG_PATH = OUTPUT_ROOT / "_overnight_log.json"

for folder in (CACHE_DIR, OUTPUT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

# ---- preflight: fail loudly and early, never eight hours in ------------------
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError("Set Accelerator -> GPU T4 x2 in Kaggle Settings, then Run All again.")
gpu_names = [torch.cuda.get_device_name(i) for i in range(2)]
print(f"GPUs: {gpu_names}")
if not all("T4" in name for name in gpu_names):
    print("Warning: timings are tuned for 2x T4.")

if not TRAIN_MANIFEST.is_file() or not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(
        f"PlantVillage input missing. Expected\n  {TRAIN_MANIFEST}\n  {IMAGE_ROOT}\n"
        "Attach the PlantVillage dataset to this notebook."
    )

dev = pd.read_csv(TRAIN_MANIFEST)
required = {"split", "relative_path", "label"}
if not required.issubset(dev.columns):
    raise ValueError(f"Development manifest needs columns {sorted(required)}, found {list(dev.columns)}")

train_df = dev.loc[dev.split == "train"].reset_index(drop=True)
val_df = dev.loc[dev.split == "validation"].reset_index(drop=True)
classes = sorted(train_df.label.unique().tolist())
if len(classes) != 38:
    raise ValueError(f"Expected 38 classes, found {len(classes)}")
if set(val_df.label) != set(classes):
    raise ValueError("Validation split does not cover the same 38 classes")
if set(train_df.relative_path) & set(val_df.relative_path):
    raise ValueError("Train and validation share image paths")

IMAGE_SIZE = 224
train_paths = [IMAGE_ROOT / "train" / rel for rel in train_df.relative_path]
val_paths = [IMAGE_ROOT / "train" / rel for rel in val_df.relative_path]
missing = [str(p) for p in train_paths[:200] + val_paths[:200] if not p.is_file()]
if missing:
    raise FileNotFoundError(f"Manifest paths do not resolve, e.g. {missing[0]}")

expected_bytes = (len(train_df) + len(val_df)) * IMAGE_SIZE * IMAGE_SIZE * 3
free_cache = shutil.disk_usage(CACHE_DIR).free
if free_cache < expected_bytes * 1.10:
    raise OSError(f"Cache needs ~{expected_bytes/1e9:.1f} GB, only {free_cache/1e9:.1f} GB free in /kaggle/temp")

def log(entry):
    """Append a progress record to disk. Defined here so no later cell depends
    on which cell ran first: an ordering dependency is exactly the kind of thing
    that kills an unattended run."""
    entries = json.loads(LOG_PATH.read_text()) if LOG_PATH.is_file() else []
    entries.append(entry)
    LOG_PATH.write_text(json.dumps(entries, indent=2), encoding="utf-8")


print(f"PlantVillage: {len(train_df):,} train / {len(val_df):,} validation / {len(classes)} classes")
print(f"Budget: {SESSION_BUDGET_H} h session, stopping new runs {RESERVE_H} h before the end")
print(f"Start time: {time.strftime('%H:%M')} — expected finish around "
      f"{time.strftime('%H:%M', time.localtime(time.time() + 6*3600))}")
print("Locked test manifest: not opened.")

## 2. Image cache

Built once and shared by all fourteen runs. Any image that fails to decode aborts the cache build instead of silently training on a black square.

In [ ]:
def cache_split(frame, paths, tag):
    labels = np.asarray([classes.index(label) for label in frame.label], dtype=np.int64)
    path_key = hashlib.sha256("\n".join(map(str, paths)).encode()).hexdigest()[:16]
    stem = CACHE_DIR / f"{tag}_{IMAGE_SIZE}_{len(paths)}_{path_key}"
    x_path, y_path, done_path = Path(f"{stem}_x.npy"), Path(f"{stem}_y.npy"), Path(f"{stem}.done")
    shape = (len(paths), IMAGE_SIZE, IMAGE_SIZE, 3)

    if done_path.is_file() and x_path.is_file() and y_path.is_file():
        try:
            existing = np.load(x_path, mmap_mode="r", allow_pickle=False)
            ok = (existing.shape == shape and existing.dtype == np.uint8
                  and np.array_equal(np.load(y_path, allow_pickle=False), labels))
            del existing
            if ok:
                print(f"Reused verified {tag} cache: {x_path}")
                return x_path, y_path
        except Exception:
            pass

    for stale in (done_path, x_path, y_path):
        stale.unlink(missing_ok=True)
    spare = shutil.disk_usage(CACHE_DIR).free - int(shape[0] * shape[1] * shape[2] * 3 * 1.05)
    if spare < 1e9:
        raise OSError(f"Not enough space to build the {tag} cache; {spare/1e9:.2f} GB would remain")

    array = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.uint8, shape=shape)
    started = time.perf_counter()
    failures = []

    def decode(item):
        i, path = item
        try:
            with Image.open(path) as image:
                array[i] = np.asarray(
                    image.convert("RGB").resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR),
                    dtype=np.uint8,
                )
            return i
        except Exception as exc:                      # keep going, report at the end
            failures.append((str(path), repr(exc)))
            array[i] = 0
            return i

    with ThreadPoolExecutor(max_workers=8) as pool:
        progress = tqdm(pool.map(decode, enumerate(paths)), total=len(paths), desc=f"Cache {tag}", unit="img")
        for count, _ in enumerate(progress, start=1):
            if count % 1000 == 0 or count == len(paths):
                progress.set_postfix(img_s=f"{count / max(time.perf_counter() - started, 1e-6):.0f}")
    array.flush()
    del array
    np.save(y_path, labels, allow_pickle=False)

    if failures:
        print(f"{len(failures)} image(s) failed to decode; first 3:")
        for path, err in failures[:3]:
            print(f"  {path}: {err}")
        raise RuntimeError(f"{len(failures)} undecodable images in the {tag} split; cache not marked complete")

    done_path.write_text("complete\n", encoding="utf-8")
    print(f"Wrote {tag} cache in {time.perf_counter() - started:.1f}s")
    return x_path, y_path


TRAIN_X_PATH, TRAIN_Y_PATH = cache_split(train_df, train_paths, "train")
VAL_X_PATH, VAL_Y_PATH = cache_split(val_df, val_paths, "val")
CLASSES_PATH = CACHE_DIR / "classes.json"
CLASSES_PATH.write_text(json.dumps(classes, indent=2), encoding="utf-8")
print("Image cache ready.")


# Defined here, next to the paths they use, so neither the gate cell nor the
# sweep cell depends on the other having run first.
cache_args = [
    "--train-images", str(TRAIN_X_PATH), "--train-labels", str(TRAIN_Y_PATH),
    "--val-images", str(VAL_X_PATH), "--val-labels", str(VAL_Y_PATH),
    "--classes-json", str(CLASSES_PATH), "--protocol", str(PROTOCOL_PATH),
]


def worker_command(arch, mode, seeds, epochs, output_root):
    return [sys.executable, str(WORKER), "--arch", arch, "--mode", mode,
            "--seeds", *[str(s) for s in seeds], "--epochs", str(epochs),
            "--output-root", str(output_root), *cache_args]

## 3. Engine, worker and protocol

Written from source, then asserted: shuffle mode, flat learning rate, no label smoothing, no class weights, and the exact reference-model settings. If any of that is wrong the notebook stops now rather than at hour six.

In [ ]:
# Writes the engine, worker and protocol, then proves all three are intact.
FILES = {
    '/kaggle/working/train_ddp.py': '#!/usr/bin/env python3\n"""Fine-tune a supported CNN on PlantVillage with two-GPU DDP.\n\nThis is the v2 training engine. It replaces the pilot\n``train_efficientnet_b0_ddp.py`` and is architecture-agnostic, so the\nEfficientNet-B0 run and the ResNet18 run share one auditable code path.\n\nWhy v2 exists (measured problems in the pilot run)\n--------------------------------------------------\n* The pilot used ``num_workers=0`` on a 5.5 GB memory-mapped cache. Every batch\n  page-faulted in the main process, so the two T4s idled while data loaded.\n  Per-epoch time grew 1.7 -> 15.65 min as the page cache evicted. v2 uses\n  worker processes with prefetch, which is the standard fix and should land\n  around 1 min/epoch.\n* The pilot set ``cudnn.benchmark = False`` and used a global batch of 32.\n  v2 enables benchmark (input shape is fixed) and defaults to 64 per GPU.\n* The loss was plain, unweighted cross-entropy on a 38.4x imbalanced training\n  set (Orange HLB 3,842 vs Potato healthy 100). v2 supports class-balanced\n  cross-entropy so macro-F1, the model-selection metric, is not starved by\n  rare classes.\n* The pilot saved only headline metrics. v2 also writes per-class\n  precision/recall/F1/support, the confusion matrix and calibration\n  diagnostics, which the report and the guide review need anyway.\n\nWhat v2 adds for accuracy\n-------------------------\nClass-balanced cross-entropy, label smoothing, cosine schedule with linear\nwarmup, gradient clipping, and an EMA copy of the weights that is evaluated and\nkept when it beats the raw weights.\n\nWhat v2 adds for safety and reproducibility\n-------------------------------------------\n``weights_only=True`` checkpoint loading with an explicit fallback, and a\ndataset fingerprint stored in the checkpoint so a resumed run cannot silently\ncontinue against a different split.\n\nUsage\n-----\n    torchrun --standalone --nproc_per_node=2 scripts/train_ddp.py \\\n        --arch efficientnet_b0 \\\n        --train-images /path/train_x.npy --train-labels /path/train_y.npy \\\n        --val-images /path/val_x.npy --val-labels /path/val_y.npy \\\n        --classes-json /path/classes.json --output-dir /path/out\n\nAdd ``--dry-run`` to execute three optimizer steps and exit, which is the\ncheap way to prove a Kaggle session is wired up before committing GPU hours.\n"""\nfrom __future__ import annotations\n\nimport os\n\n# Keep BLAS single-threaded per rank: DDP already saturates the cores and\n# oversubscription is a common cause of stalled steps.\nos.environ.setdefault("OMP_NUM_THREADS", "1")\nos.environ.setdefault("MKL_NUM_THREADS", "1")\n\nimport argparse\nimport contextlib\nimport copy\nimport hashlib\nimport json\nimport math\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.distributed as dist\nimport torch.nn.functional as F\nfrom sklearn.metrics import (\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_recall_fscore_support,\n    top_k_accuracy_score,\n)\nfrom torch import nn\nfrom torch.nn.parallel import DistributedDataParallel as DDP\nfrom torch.utils.data import DataLoader, Dataset, Subset\nfrom tqdm.auto import tqdm\n\ntry:  # torchvision is a hard dependency in practice; import lazily for CLI help\n    import torchvision\n    from torchvision import models\nexcept Exception as exc:  # pragma: no cover - only hit in broken installs\n    torchvision = None\n    models = None\n    _TV_IMPORT_ERROR = exc\n\nIMAGENET_MEAN = (0.485, 0.456, 0.406)\nIMAGENET_STD = (0.229, 0.224, 0.225)\nSMALLCNN_MEAN = (0.5, 0.5, 0.5)\nSMALLCNN_STD = (0.5, 0.5, 0.5)\n\nARCHES = ("smallcnn", "resnet18", "efficientnet_b0", "convnext_tiny")\n\n\n# --------------------------------------------------------------------------- #\n# Data\n# --------------------------------------------------------------------------- #\nclass CachedImages(Dataset):\n    """Read one uint8 image at a time from a disk-backed NPY memory map."""\n\n    def __init__(self, image_file, labels_file, image_size: int):\n        self.images = np.load(image_file, mmap_mode="r", allow_pickle=False)\n        self.labels = np.load(labels_file, mmap_mode="r", allow_pickle=False)\n        expected = (image_size, image_size, 3)\n        if self.images.ndim != 4 or tuple(self.images.shape[1:]) != expected:\n            raise ValueError(f"Bad cached image shape: {self.images.shape}, expected (*, {expected})")\n        if self.images.dtype != np.uint8:\n            raise ValueError(f"Image cache must be uint8, found {self.images.dtype}")\n        if len(self.images) != len(self.labels):\n            raise ValueError("Image cache and label cache have different lengths")\n\n    def __len__(self) -> int:\n        return len(self.labels)\n\n    def __getitem__(self, index: int):\n        # copy() detaches the row from the memory map so the loader can batch it\n        image = torch.from_numpy(self.images[index].copy())\n        return image, int(self.labels[index])\n\n\nclass SmallCNN(nn.Module):\n    """Four-stage CNN baseline.\n\n    Defined here as well as in src/plant_disease/models/smallcnn.py because this\n    file has to run standalone inside Kaggle, where the package is not installed.\n    scripts/smoke_test_engine.py asserts the two definitions are parameter-\n    identical, so they cannot drift apart silently.\n    """\n\n    def __init__(self, num_classes: int):\n        super().__init__()\n        blocks = []\n        channels = 3\n        for output_channels in (32, 64, 128, 256):\n            blocks.extend([\n                nn.Conv2d(channels, output_channels, 3, padding=1, bias=False),\n                nn.BatchNorm2d(output_channels),\n                nn.ReLU(inplace=True),\n                nn.MaxPool2d(2),\n            ])\n            channels = output_channels\n        self.features = nn.Sequential(*blocks)\n        self.head = nn.Sequential(\n            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.30), nn.Linear(256, num_classes)\n        )\n\n    def forward(self, images):\n        return self.head(self.features(images))\n\n\ndef build_model(arch: str, num_classes: int, pretrained: bool) -> nn.Module:\n    if arch == "smallcnn":\n        # Random initialisation by design; there is no pretrained SmallCNN.\n        return SmallCNN(num_classes)\n    if models is None:  # pragma: no cover\n        raise RuntimeError(f"torchvision import failed: {_TV_IMPORT_ERROR}")\n    if arch == "efficientnet_b0":\n        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None\n        model = models.efficientnet_b0(weights=weights)\n        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, num_classes)\n    elif arch == "resnet18":\n        weights = models.ResNet18_Weights.DEFAULT if pretrained else None\n        model = models.resnet18(weights=weights)\n        model.fc = nn.Linear(model.fc.in_features, num_classes)\n    elif arch == "convnext_tiny":\n        weights = models.ConvNeXt_Tiny_Weights.DEFAULT if pretrained else None\n        model = models.convnext_tiny(weights=weights)\n        model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)\n    else:\n        raise ValueError(f"Unsupported arch {arch!r}; choose from {ARCHES}")\n    return model\n\n\ndef dataset_fingerprint(train_labels, val_labels, classes, image_size: int) -> str:\n    """Stable identifier for the exact split a checkpoint was trained on."""\n    digest = hashlib.sha256()\n    digest.update(np.ascontiguousarray(train_labels).tobytes())\n    digest.update(np.ascontiguousarray(val_labels).tobytes())\n    digest.update(json.dumps(classes, separators=(",", ":")).encode("utf-8"))\n    digest.update(str(image_size).encode("utf-8"))\n    return digest.hexdigest()\n\n\ndef block_shuffled_indices(n: int, rng: np.random.Generator, block: int = 2048) -> np.ndarray:\n    """Permute block order, then shuffle inside each block.\n\n    Reading in blocks keeps the page cache local instead of jumping across the\n    whole 5 GB map every batch, which was the pilot\'s failure mode.\n    """\n    order = np.arange(n)\n    starts = np.arange(0, n, block)\n    rng.shuffle(starts)\n    out = []\n    for start in starts:\n        chunk = order[start:start + block]\n        rng.shuffle(chunk)\n        out.append(chunk)\n    return np.concatenate(out)\n\n\n# --------------------------------------------------------------------------- #\n# Augmentation and normalisation (all on GPU)\n# --------------------------------------------------------------------------- #\ndef amp_context(use_cuda: bool):\n    """FP16 autocast on the accelerator, no-op on CPU (CPU autocast is slower)."""\n    if use_cuda:\n        return torch.amp.autocast(device_type="cuda", dtype=torch.float16)\n    return contextlib.nullcontext()\n\n\ndef gpu_augment_and_normalize(image_u8, device, training: bool, mean, std, rotation_deg: float = 12.0):\n    """NHWC uint8 -> normalised NCHW float, with augmentation applied on the GPU.\n\n    Augmentation runs on the accelerator so no CPU worker is the bottleneck and\n    the worker processes only have to memcpy bytes out of the memory map.\n    """\n    x = image_u8.to(device=device, dtype=torch.float32, non_blocking=True)\n    x = x.permute(0, 3, 1, 2).div_(255.0).contiguous(memory_format=torch.channels_last)\n    if training:\n        batch = x.shape[0]\n        # horizontal flip\n        flip = torch.rand(batch, device=device) < 0.5\n        x = torch.where(flip.view(batch, 1, 1, 1), x.flip(3), x)\n        # small rotation\n        if rotation_deg > 0:\n            angle = (torch.rand(batch, device=device) * 2.0 - 1.0) * math.radians(rotation_deg)\n            cosine, sine = angle.cos(), angle.sin()\n            theta = torch.zeros(batch, 2, 3, device=device)\n            theta[:, 0, 0], theta[:, 0, 1] = cosine, -sine\n            theta[:, 1, 0], theta[:, 1, 1] = sine, cosine\n            grid = F.affine_grid(theta, x.shape, align_corners=False)\n            x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)\n        # brightness and per-channel contrast jitter\n        brightness = torch.rand(batch, 1, 1, 1, device=device) * 0.24 + 0.88\n        x = (x * brightness).clamp_(0.0, 1.0)\n        gain = torch.rand(batch, 3, 1, 1, device=device) * 0.16 + 0.92\n        x = (x * gain).clamp_(0.0, 1.0)\n    mean_t = torch.tensor(mean, device=device).view(1, 3, 1, 1)\n    std_t = torch.tensor(std, device=device).view(1, 3, 1, 1)\n    return ((x - mean_t) / std_t).contiguous(memory_format=torch.channels_last)\n\n\n# --------------------------------------------------------------------------- #\n# Metrics and calibration\n# --------------------------------------------------------------------------- #\ndef expected_calibration_error(probabilities: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:\n    """Binned |accuracy - confidence|, weighted by bin population."""\n    confidence = probabilities.max(axis=1)\n    predictions = probabilities.argmax(axis=1)\n    correct = (predictions == y_true).astype(np.float64)\n    edges = np.linspace(0.0, 1.0, n_bins + 1)\n    ece = 0.0\n    for low, high in zip(edges[:-1], edges[1:]):\n        mask = (confidence > low) & (confidence <= high)\n        if mask.any():\n            ece += mask.mean() * abs(correct[mask].mean() - confidence[mask].mean())\n    return float(ece)\n\n\ndef brier_score(probabilities: np.ndarray, y_true: np.ndarray) -> float:\n    """Mean squared error between predicted probabilities and one-hot labels."""\n    one_hot = np.zeros_like(probabilities)\n    one_hot[np.arange(len(y_true)), y_true] = 1.0\n    return float(np.mean(np.sum((probabilities - one_hot) ** 2, axis=1)))\n\n\ndef score_predictions(y_true, y_pred, probabilities=None) -> dict:\n    y_true = np.asarray(y_true)\n    y_pred = np.asarray(y_pred)\n    metrics = {\n        "accuracy": float((y_true == y_pred).mean()),\n        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),\n        "macro_precision": float(precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)[0]),\n        "macro_recall": float(precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)[1]),\n        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),\n        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),\n    }\n    if probabilities is not None:\n        scores = probabilities.astype(np.float64)\n        labels = np.arange(scores.shape[1])\n        metrics["top3_accuracy"] = float(top_k_accuracy_score(y_true, scores, k=3, labels=labels))\n        metrics["ece"] = expected_calibration_error(scores, y_true)\n        metrics["brier"] = brier_score(scores, y_true)\n        metrics["mean_confidence"] = float(scores.max(axis=1).mean())\n        metrics["frac_below_0_5"] = float((scores.max(axis=1) < 0.5).mean())\n    return metrics\n\n\ndef classification_table(y_true, y_pred, classes) -> pd.DataFrame:\n    precision, recall, f1, support = precision_recall_fscore_support(\n        y_true, y_pred, labels=np.arange(len(classes)), zero_division=0\n    )\n    return pd.DataFrame(\n        {\n            "class": classes,\n            "precision": precision,\n            "recall": recall,\n            "f1": f1,\n            "support": support,\n        }\n    )\n\n\n# --------------------------------------------------------------------------- #\n# EMA\n# --------------------------------------------------------------------------- #\nclass ModelEma:\n    """Exponential moving average of model weights, kept on the evaluating rank."""\n\n    def __init__(self, model: nn.Module, decay: float = 0.999):\n        self.module = copy.deepcopy(model).eval()\n        for parameter in self.module.parameters():\n            parameter.requires_grad_(False)\n        self.decay = decay\n        self.updates = 0\n\n    @torch.no_grad()\n    def update(self, model: nn.Module):\n        self.updates += 1\n        # Ramp the effective decay early so the EMA is not anchored to the\n        # random initialisation for the first few hundred steps.\n        decay = min(self.decay, (1 + self.updates) / (10 + self.updates))\n        for ema_parameter, parameter in zip(self.module.parameters(), model.parameters()):\n            ema_parameter.mul_(decay).add_(parameter.detach(), alpha=1.0 - decay)\n        for ema_buffer, buffer in zip(self.module.buffers(), model.buffers()):\n            ema_buffer.copy_(buffer)\n\n\n# --------------------------------------------------------------------------- #\n# Evaluation\n# --------------------------------------------------------------------------- #\n@torch.no_grad()\ndef evaluate(model, dataset, device, mean, std, batch_size, workers, use_cuda=True) -> tuple:\n    model.eval()\n    loader = DataLoader(\n        dataset,\n        batch_size=batch_size,\n        shuffle=False,\n        num_workers=workers,\n        pin_memory=use_cuda,\n        persistent_workers=workers > 0,\n    )\n    y_true, y_pred, probability_rows = [], [], []\n    started = time.perf_counter()\n    progress = tqdm(loader, desc="validation", unit="batch", leave=True, mininterval=1.0, disable=None)\n    for images, labels in progress:\n        x = gpu_augment_and_normalize(images, device, training=False, mean=mean, std=std)\n        with amp_context(use_cuda):\n            logits = model(x)\n        probabilities = torch.softmax(logits.float(), dim=1).cpu().numpy()\n        y_true.extend(labels.numpy().tolist())\n        y_pred.extend(probabilities.argmax(axis=1).tolist())\n        probability_rows.append(probabilities)\n        seen = len(y_true)\n        progress.set_postfix(\n            val_accuracy=f"{np.mean(np.asarray(y_true) == np.asarray(y_pred)):.4f}",\n            img_s=f"{seen / max(time.perf_counter() - started, 1e-6):.1f}",\n            refresh=False,\n        )\n    probabilities = np.concatenate(probability_rows, axis=0)\n    return score_predictions(y_true, y_pred, probabilities), np.asarray(y_true), np.asarray(y_pred), probabilities\n\n\n# --------------------------------------------------------------------------- #\n# Checkpoint IO\n# --------------------------------------------------------------------------- #\ndef atomic_save(payload, path: Path):\n    temporary = path.with_name(path.name + ".tmp")\n    torch.save(payload, temporary)\n    os.replace(temporary, path)\n\n\ndef safe_load(path: Path, map_location):\n    """Load a checkpoint, preferring the non-executing ``weights_only`` path."""\n    try:\n        return torch.load(path, map_location=map_location, weights_only=True)\n    except Exception as exc:  # older torch or an exotic payload\n        print(f"  weights_only=True failed for {path.name} ({exc}); retrying in compatibility mode", flush=True)\n        return torch.load(path, map_location=map_location, weights_only=False)\n\n\ndef clean_state_dict(state):\n    if state and all(str(key).startswith("module.") for key in state):\n        return {key[7:]: value for key, value in state.items()}\n    return state\n\n\n# --------------------------------------------------------------------------- #\n# Main\n# --------------------------------------------------------------------------- #\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--train-images", required=True)\n    parser.add_argument("--train-labels", required=True)\n    parser.add_argument("--val-images", required=True)\n    parser.add_argument("--val-labels", required=True)\n    parser.add_argument("--classes-json", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--arch", default="efficientnet_b0", choices=ARCHES)\n    parser.add_argument("--epochs", type=int, default=12)\n    parser.add_argument("--patience", type=int, default=3)\n    parser.add_argument("--local-batch-size", type=int, default=64, help="per-GPU batch; global = this x world size")\n    parser.add_argument("--eval-batch-size", type=int, default=128)\n    parser.add_argument("--lr", type=float, default=3e-4)\n    parser.add_argument("--min-lr-factor", type=float, default=0.01)\n    parser.add_argument("--weight-decay", type=float, default=0.05)\n    parser.add_argument("--warmup-fraction", type=float, default=0.10)\n    parser.add_argument("--label-smoothing", type=float, default=0.05)\n    parser.add_argument("--grad-clip", type=float, default=5.0)\n    parser.add_argument("--class-weight-alpha", type=float, default=0.5,\n                        help="0 disables class-balanced loss; 1 is full inverse frequency")\n    parser.add_argument("--ema-decay", type=float, default=0.999, help="0 disables EMA")\n    parser.add_argument("--rotation-deg", type=float, default=12.0)\n    parser.add_argument("--workers", type=int, default=4, help="DataLoader workers per rank")\n    parser.add_argument("--seed", type=int, default=42)\n    parser.add_argument("--image-size", type=int, default=224)\n    parser.add_argument("--compile", action="store_true", help="torch.compile the model (slower first step)")\n    parser.add_argument("--shuffle-mode", default="blocked", choices=("random", "blocked"))\n    parser.add_argument("--dry-run", action="store_true", help="run 3 optimizer steps then exit")\n    parser.add_argument("--device", default="auto", choices=("auto", "cuda", "cpu"),\n                        help="auto uses CUDA when present; cpu exists for local smoke tests")\n    # Escape hatch for environments where torchrun\'s TCP store is unavailable\n    # (some Windows CPU wheels are built without libuv). A file-based store keeps\n    # the same code path reachable without the launcher.\n    parser.add_argument("--init-method", default="env://",\n                        help="URL for process-group init; non-env values require --rank/--world-size")\n    parser.add_argument("--rank", type=int, default=None)\n    parser.add_argument("--world-size", type=int, default=None)\n    parser.add_argument("--no-dist", action="store_true",\n                        help="single process, no process group; for local code checks only")\n    args = parser.parse_args()\n\n    use_cuda = args.device == "cuda" or (args.device == "auto" and torch.cuda.is_available())\n    if args.device == "cuda" and not torch.cuda.is_available():\n        raise RuntimeError("CUDA was requested but is unavailable. In Kaggle select Accelerator -> GPU T4 x2.")\n    if not use_cuda and args.device == "auto":\n        raise RuntimeError(\n            "No CUDA device. Pass --device cpu to run the engine on CPU (slow, intended for a "\n            "small local check) or select GPU T4 x2 in Kaggle."\n        )\n\n    backend = "nccl" if use_cuda else "gloo"\n    distributed = not args.no_dist\n    if not distributed:\n        rank, world, local_rank = 0, 1, 0\n    elif args.init_method == "env://":\n        dist.init_process_group(backend=backend)\n        rank = dist.get_rank()\n        world = dist.get_world_size()\n        local_rank = int(os.environ.get("LOCAL_RANK", 0))\n    else:\n        if args.rank is None or args.world_size is None:\n            raise ValueError("--init-method other than env:// requires --rank and --world-size")\n        dist.init_process_group(\n            backend=backend, init_method=args.init_method, rank=args.rank, world_size=args.world_size\n        )\n        rank, world, local_rank = args.rank, args.world_size, args.rank\n\n    def synchronise():\n        """Barrier when a process group exists, otherwise a no-op."""\n        if distributed:\n            dist.barrier()\n\n    def unwrap(module: nn.Module) -> nn.Module:\n        return module.module if isinstance(module, DDP) else module\n    if use_cuda:\n        torch.cuda.set_device(local_rank)\n        device = torch.device("cuda", local_rank)\n    else:\n        device = torch.device("cpu")\n    torch.set_num_threads(1)\n    random.seed(args.seed + rank)\n    np.random.seed(args.seed + rank)\n    torch.manual_seed(args.seed + rank)\n    if use_cuda:\n        torch.cuda.manual_seed_all(args.seed + rank)\n        # Fixed 224x224 input means cuDNN can pick and keep the fastest algorithm.\n        torch.backends.cudnn.benchmark = True\n        torch.backends.cuda.matmul.allow_tf32 = True\n    torch.set_float32_matmul_precision("high")\n\n    classes = json.loads(Path(args.classes_json).read_text(encoding="utf-8"))\n    train_ds = CachedImages(args.train_images, args.train_labels, args.image_size)\n    val_ds = CachedImages(args.val_images, args.val_labels, args.image_size)\n    if len(train_ds) == 0 or len(val_ds) == 0:\n        raise ValueError("Both train and validation caches must contain images")\n    if int(np.asarray(train_ds.labels).max()) >= len(classes) or int(np.asarray(val_ds.labels).max()) >= len(classes):\n        raise ValueError("A cached label index is out of range for the class list")\n\n    fingerprint = dataset_fingerprint(\n        np.asarray(train_ds.labels), np.asarray(val_ds.labels), classes, args.image_size\n    )\n\n    mean, std = (SMALLCNN_MEAN, SMALLCNN_STD) if args.arch == "smallcnn" else (IMAGENET_MEAN, IMAGENET_STD)\n\n    output_dir = Path(args.output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    best_path = output_dir / f"{args.arch}_best.pt"\n    last_path = output_dir / f"{args.arch}_last.pt"\n\n    resume_config = {\n        "architecture": args.arch,\n        "classes": classes,\n        "seed": args.seed,\n        "image_size": args.image_size,\n        "local_batch_size": args.local_batch_size,\n        "global_batch_size": args.local_batch_size * world,\n        "optimizer": "AdamW",\n        "learning_rate": args.lr,\n        "weight_decay": args.weight_decay,\n        "label_smoothing": args.label_smoothing,\n        "class_weight_alpha": args.class_weight_alpha,\n        "ema_decay": args.ema_decay,\n        "max_epochs": args.epochs,\n        "patience": args.patience,\n        "dataset_fingerprint": fingerprint,\n        "torch_version": str(torch.__version__),\n        "torchvision_version": str(torchvision.__version__),\n    }\n\n    resume = None\n    if last_path.is_file() and not args.dry_run:\n        resume = safe_load(last_path, "cpu")\n        stored = resume.get("resume_config", {})\n        if stored.get("dataset_fingerprint") not in (None, fingerprint):\n            raise RuntimeError(\n                "The resume checkpoint was trained on a different split "\n                f"({stored.get(\'dataset_fingerprint\')} != {fingerprint}). "\n                "Move the old run aside or use a fresh --output-dir."\n            )\n        if {k: v for k, v in stored.items() if k != "dataset_fingerprint"} != {\n            k: v for k, v in resume_config.items() if k != "dataset_fingerprint"\n        }:\n            raise RuntimeError("Resume checkpoint settings differ from this run. Use a fresh --output-dir.")\n        if rank == 0:\n            print(f"Resuming from completed epoch {resume[\'epoch\']}.", flush=True)\n\n    # Class-balanced loss, softened by --class-weight-alpha. alpha=0 is plain CE.\n    train_labels = np.asarray(train_ds.labels)\n    class_counts = np.bincount(train_labels, minlength=len(classes)).astype(np.float64)\n    if args.class_weight_alpha > 0:\n        present = class_counts > 0\n        weights = np.ones(len(classes), dtype=np.float64)\n        weights[present] = (class_counts.sum() / (len(classes) * class_counts[present])) ** args.class_weight_alpha\n        weights = weights / weights[present].mean()\n    else:\n        weights = np.ones(len(classes), dtype=np.float64)\n    loss_weight = torch.tensor(weights, dtype=torch.float32, device=device)\n\n    pretrained = rank == 0 and resume is None\n    base = build_model(args.arch, len(classes), pretrained=pretrained)\n    base = base.to(device=device, memory_format=torch.channels_last)\n    if resume is not None:\n        base.load_state_dict(clean_state_dict(resume["model_state_dict"]))\n    # Broadcast rank 0\'s weights *and* buffers so every rank starts identical.\n    # Buffers matter for transfer learning: batch-norm running statistics on the\n    # non-zero ranks would otherwise stay at their default (mean 0, var 1).\n    if distributed:\n        for parameter in base.parameters():\n            dist.broadcast(parameter.data, src=0)\n        for buffer in base.buffers():\n            dist.broadcast(buffer.data, src=0)\n\n    # Build the EMA from the plain module, before torch.compile and DDP wrap it.\n    ema = ModelEma(base, decay=args.ema_decay) if (args.ema_decay > 0 and rank == 0) else None\n\n    if args.compile:\n        try:\n            base = torch.compile(base)\n            if rank == 0:\n                print("torch.compile enabled.", flush=True)\n        except Exception as exc:\n            print(f"torch.compile unavailable, continuing eagerly: {exc}", flush=True)\n\n    model = (\n        DDP(base, device_ids=[local_rank], output_device=local_rank,\n            broadcast_buffers=True, find_unused_parameters=False)\n        if distributed else base\n    )\n    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)\n    # A disabled scaler still needs a valid device type, so name CPU explicitly\n    # when there is no accelerator rather than constructing a CUDA object.\n    try:\n        scaler = torch.amp.GradScaler("cuda" if use_cuda else "cpu", enabled=use_cuda)\n    except (TypeError, RuntimeError):  # torch < 2.3 positional signature\n        scaler = torch.cuda.amp.GradScaler(enabled=use_cuda)\n\n    steps_per_epoch = max(len(train_ds) // (args.local_batch_size * world), 1)\n    total_steps = steps_per_epoch * args.epochs\n    warmup_steps = max(int(total_steps * args.warmup_fraction), 1)\n\n    def lr_lambda(step: int) -> float:\n        if step < warmup_steps:\n            return (step + 1) / warmup_steps\n        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)\n        cosine = 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))\n        return args.min_lr_factor + (1.0 - args.min_lr_factor) * cosine\n\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)\n\n    start_epoch, history, best_f1, best_epoch, stale, prior_seconds = 1, [], -1.0, 0, 0, 0.0\n    global_step = 0\n    if resume is not None:\n        loadable = {k: v for k, v in resume.items() if k not in {"model_state_dict", "resume_config"}}\n        optimizer.load_state_dict(loadable["optimizer_state_dict"])\n        scaler.load_state_dict(loadable["scaler_state_dict"])\n        if "scheduler_state_dict" in loadable:\n            scheduler.load_state_dict(loadable["scheduler_state_dict"])\n        if ema is not None and loadable.get("ema_state_dict"):\n            ema.module.load_state_dict(loadable["ema_state_dict"])\n            ema.updates = int(loadable.get("ema_updates", 0))\n        start_epoch = int(resume["epoch"]) + 1\n        history = resume.get("history", []) if rank == 0 else []\n        best_f1 = float(resume.get("best_val_macro_f1", -1.0))\n        best_epoch = int(resume.get("best_epoch", 0))\n        stale = int(resume.get("stale", 0))\n        prior_seconds = float(resume.get("elapsed_seconds", 0.0))\n        global_step = int(resume.get("global_step", 0))\n        if stale >= args.patience:\n            start_epoch = args.epochs + 1\n            if rank == 0:\n                print("Early stopping was already reached; using the saved best checkpoint.", flush=True)\n\n    if rank == 0:\n        gpu_label = (\n            ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count()))\n            if use_cuda else "cpu"\n        )\n        print(\n            f"{args.arch} | devices={gpu_label} | "\n            f"global batch={args.local_batch_size * world} ({args.local_batch_size}/GPU) | "\n            f"workers/rank={args.workers} | lr={args.lr} | train={len(train_ds):,} | "\n            f"val={len(val_ds):,} | classes={len(classes)} | steps/epoch={steps_per_epoch}",\n            flush=True,\n        )\n        print(f"Dataset fingerprint: {fingerprint[:16]}...", flush=True)\n\n    started = time.time()\n    criterion = nn.CrossEntropyLoss(weight=loss_weight, label_smoothing=args.label_smoothing)\n\n    for epoch in range(start_epoch, args.epochs + 1):\n        rng = np.random.default_rng(args.seed + epoch)\n        if args.shuffle_mode == "blocked":\n            order = block_shuffled_indices(len(train_ds), rng)\n        else:\n            order = rng.permutation(len(train_ds))\n        rank_indices = order[rank::world].tolist()\n\n        train_loader = DataLoader(\n            Subset(train_ds, rank_indices),\n            batch_size=args.local_batch_size,\n            shuffle=False,\n            drop_last=True,\n            num_workers=args.workers,\n            pin_memory=use_cuda,\n            persistent_workers=args.workers > 0,\n            prefetch_factor=4 if args.workers > 0 else None,\n        )\n        model.train()\n        epoch_loss = torch.zeros((), device=device)\n        images_seen = 0\n        epoch_started = time.perf_counter()\n        progress = tqdm(\n            total=len(train_loader),\n            desc=f"{args.arch} epoch {epoch}/{args.epochs} train",\n            unit="batch",\n            leave=True,\n            mininterval=1.0,\n            disable=(rank != 0),\n        )\n        for step, (images, labels) in enumerate(train_loader, 1):\n            x = gpu_augment_and_normalize(\n                images, device, training=True, mean=mean, std=std, rotation_deg=args.rotation_deg\n            )\n            y = labels.to(device=device, non_blocking=True)\n            optimizer.zero_grad(set_to_none=True)\n            with amp_context(use_cuda):\n                logits = model(x)\n                loss = criterion(logits, y)\n            scaler.scale(loss).backward()\n            if args.grad_clip > 0:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            global_step += 1\n            if ema is not None:\n                ema.update(unwrap(model)._orig_mod if hasattr(unwrap(model), "_orig_mod") else unwrap(model))\n            epoch_loss += loss.detach()\n            images_seen += len(y)\n            if rank == 0:\n                progress.update(1)\n                if step % 20 == 0 or step == len(train_loader):\n                    elapsed = max(time.perf_counter() - epoch_started, 1e-6)\n                    progress.set_postfix(\n                        loss=f"{loss.detach().item():.4f}",\n                        avg_loss=f"{(epoch_loss.detach().item() / step):.4f}",\n                        img_s=f"{images_seen * world / elapsed:.1f}",\n                        lr=f"{scheduler.get_last_lr()[0]:.2e}",\n                        refresh=True,\n                    )\n            if args.dry_run and step >= 3:\n                break\n        progress.close()\n        if distributed:\n            dist.all_reduce(epoch_loss, op=dist.ReduceOp.SUM)\n        synchronise()\n\n        if args.dry_run:\n            if rank == 0:\n                print("Dry run complete: 3 steps executed, no checkpoint written.", flush=True)\n            synchronise()\n            if distributed:\n                dist.destroy_process_group()\n            return\n\n        report = [None]\n        if rank == 0:\n            train_loss = float(epoch_loss.detach().item() / max(len(train_loader) * world, 1))\n\n            raw_metrics, y_true, y_pred, _ = evaluate(\n                unwrap(model), val_ds, device, mean, std, args.eval_batch_size, args.workers, use_cuda\n            )\n            chosen_metrics, chosen_source = raw_metrics, "raw"\n            if ema is not None:\n                ema_metrics, ema_true, ema_pred, _ = evaluate(\n                    ema.module, val_ds, device, mean, std, args.eval_batch_size, args.workers, use_cuda\n                )\n                if ema_metrics["macro_f1"] > raw_metrics["macro_f1"]:\n                    chosen_metrics, chosen_source = ema_metrics, "ema"\n                    y_true, y_pred = ema_true, ema_pred\n\n            # Record the applied learning rate. Without this the schedule is\n            # invisible in the artifacts: protocol v1 decayed to nearly zero\n            # every run and nothing on disk said so.\n            epoch_lr = scheduler.get_last_lr()[0] if scheduler is not None else args.lr\n\n            row = {\n                "model": args.arch,\n                "epoch": epoch,\n                "lr": epoch_lr,\n                "train_loss": train_loss,\n                **{f"val_{key}": value for key, value in raw_metrics.items()},\n                f"val_macro_f1_{chosen_source}": chosen_metrics["macro_f1"],\n                "weights_used": chosen_source,\n                "elapsed_minutes": round((prior_seconds + time.time() - started) / 60.0, 2),\n            }\n            history.append(row)\n\n            if chosen_metrics["macro_f1"] > best_f1:\n                best_f1, best_epoch, stale = chosen_metrics["macro_f1"], epoch, 0\n                state = ema.module.state_dict() if chosen_source == "ema" else unwrap(model).state_dict()\n                atomic_save(\n                    {\n                        "architecture": args.arch,\n                        "model_state_dict": clean_state_dict(state),\n                        "classes": classes,\n                        "image_size": args.image_size,\n                        "pretrained": True,\n                        "normalization_mean": list(mean),\n                        "normalization_std": list(std),\n                        "epoch": epoch,\n                        "val_macro_f1": best_f1,\n                        "weights_are_ema": chosen_source == "ema",\n                        "seed": args.seed,\n                        "dataset_fingerprint": fingerprint,\n                        "n_train": len(train_ds),\n                        "n_val": len(val_ds),\n                    },\n                    best_path,\n                )\n                pd.DataFrame(classification_table(y_true, y_pred, classes)).to_csv(\n                    output_dir / "validation_classification_report.csv", index=False\n                )\n                pd.DataFrame(\n                    confusion_matrix(y_true, y_pred, labels=np.arange(len(classes))),\n                    index=classes,\n                    columns=classes,\n                ).to_csv(output_dir / "validation_confusion_matrix.csv")\n                (output_dir / "validation_metrics.json").write_text(\n                    json.dumps({**chosen_metrics, "weights_used": chosen_source, "epoch": epoch}, indent=2),\n                    encoding="utf-8",\n                )\n            else:\n                stale += 1\n\n            elapsed_seconds = prior_seconds + time.time() - started\n            atomic_save(\n                {\n                    "architecture": args.arch,\n                    "model_state_dict": clean_state_dict(unwrap(model).state_dict()),\n                    "optimizer_state_dict": optimizer.state_dict(),\n                    "scaler_state_dict": scaler.state_dict(),\n                    "scheduler_state_dict": scheduler.state_dict(),\n                    "ema_state_dict": clean_state_dict(ema.module.state_dict()) if ema is not None else None,\n                    "ema_updates": ema.updates if ema is not None else 0,\n                    "classes": classes,\n                    "epoch": epoch,\n                    "best_val_macro_f1": best_f1,\n                    "best_epoch": best_epoch,\n                    "stale": stale,\n                    "history": history,\n                    "global_step": global_step,\n                    "elapsed_seconds": elapsed_seconds,\n                    "dataset_fingerprint": fingerprint,\n                    "resume_config": resume_config,\n                },\n                last_path,\n            )\n            pd.DataFrame(history).to_csv(output_dir / "training_history.csv", index=False)\n            print(\n                f"Epoch {epoch}/{args.epochs} | loss={train_loss:.4f} | "\n                f"val_acc={raw_metrics[\'accuracy\']:.4f} | val_macro_f1={raw_metrics[\'macro_f1\']:.4f} | "\n                f"val_f1_used={chosen_metrics[\'macro_f1\']:.4f} ({chosen_source}) | "\n                f"ece={raw_metrics.get(\'ece\', float(\'nan\')):.4f} | best_epoch={best_epoch} | "\n                f"elapsed={elapsed_seconds / 60.0:.2f} min",\n                flush=True,\n            )\n            report[0] = {\n                "best_f1": best_f1,\n                "best_epoch": best_epoch,\n                "stale": stale,\n                "stop": stale >= args.patience,\n                "elapsed_seconds": elapsed_seconds,\n            }\n        if distributed:\n            dist.broadcast_object_list(report, src=0, device=device)\n        best_f1, best_epoch, stale = report[0]["best_f1"], report[0]["best_epoch"], report[0]["stale"]\n        prior_seconds = report[0]["elapsed_seconds"]\n        if report[0]["stop"]:\n            if rank == 0:\n                print(f"Early stopping: validation macro-F1 did not improve for {args.patience} epochs.", flush=True)\n            break\n\n    synchronise()\n    if rank == 0:\n        if not best_path.is_file():\n            raise FileNotFoundError("No best checkpoint was written; training never reached a validation pass.")\n        best = safe_load(best_path, device)\n        final = build_model(args.arch, len(classes), pretrained=False).to(device)\n        final.load_state_dict(clean_state_dict(best["model_state_dict"]))\n        final.eval()\n        final_metrics, y_true, y_pred, _ = evaluate(final, val_ds, device, mean, std, args.eval_batch_size, args.workers, use_cuda)\n        summary = {\n            "model": args.arch,\n            "pretrained": bool(pretrained),\n            "validation_images": len(val_ds),\n            "train_images": len(train_ds),\n            **{f"val_{key}": value for key, value in final_metrics.items()},\n            "best_epoch": best_epoch,\n            "weights_are_ema": bool(best.get("weights_are_ema")),\n            "parameters": sum(p.numel() for p in final.parameters()),\n            "checkpoint_mb": round(best_path.stat().st_size / (1024 ** 2), 2),\n            "train_minutes": round(prior_seconds / 60.0, 2),\n            "training_mode": "imagenet_finetune_ddp_v2",\n            "used_gpus": world,\n            "local_batch_size": args.local_batch_size,\n            "batch_size": args.local_batch_size * world,\n            "global_batch_size": args.local_batch_size * world,\n            "workers_per_rank": args.workers,\n            "learning_rate": args.lr,\n            "weight_decay": args.weight_decay,\n            "rotation_deg": args.rotation_deg,\n            "warmup_fraction": args.warmup_fraction,\n            "min_lr_factor": args.min_lr_factor,\n            "max_epochs": args.epochs,\n            "patience": args.patience,\n            "label_smoothing": args.label_smoothing,\n            "class_weight_alpha": args.class_weight_alpha,\n            "ema_decay": args.ema_decay,\n            "checkpoint": str(best_path),\n            "dataset_fingerprint": fingerprint,\n        }\n        (output_dir / "validation_metrics.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")\n        print("Training complete:", json.dumps(summary, indent=2), flush=True)\n    synchronise()\n    if distributed:\n        dist.destroy_process_group()\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/sweep_worker.py': '#!/usr/bin/env python3\n"""Run one architecture over several seeds, on either one GPU or both.\n\nThis exists so a sweep can be scheduled instead of run one-architecture-per-\nsession. It shells out to ``train_ddp.py`` once per seed, which keeps every run\nan independent process with its own resumable checkpoint.\n\nTwo modes:\n\n``--mode ddp``\n    Uses both GPUs through ``torch.distributed.run``. Right for the models that\n    are bandwidth-bound on a single T4 (EfficientNet-B0, ConvNeXt-Tiny).\n\n``--mode single``\n    One process, no process group, pinned with ``CUDA_VISIBLE_DEVICES``. Right\n    for the small models, where two GPUs per model waste half the machine. Two\n    of these can run side by side, one per GPU.\n\nWhy the split matters: the pilot measured SmallCNN at 0.67 min/epoch and\nEfficientNet-B0 at 2.45 min/epoch. Running SmallCNN on both GPUs to finish in\n40 seconds is worse than running it on one GPU while something else uses the\nother.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nENGINE = HERE / "train_ddp.py"\n\n\ndef build_command(arch, seed, cfg, protocol, paths, out_dir, mode, script, device, epochs):\n    # Arguments only. The interpreter and the script path are added once, by the\n    # mode branch below: torchrun takes the script as its first positional\n    # argument, so leaving "python" in here made torchrun try to execute the\n    # interpreter binary as Python source ("source code cannot contain null\n    # bytes" / "ELF" in the traceback).\n    args = [\n        "--arch", arch,\n        "--train-images", str(paths["train_x"]), "--train-labels", str(paths["train_y"]),\n        "--val-images", str(paths["val_x"]), "--val-labels", str(paths["val_y"]),\n        "--classes-json", str(paths["classes"]), "--output-dir", str(out_dir),\n        "--epochs", str(epochs), "--patience", str(protocol["patience"]),\n        "--local-batch-size", str(cfg["local_batch_size"]),\n        "--eval-batch-size", str(protocol["eval_batch_size"]),\n        "--lr", str(cfg["lr"]), "--weight-decay", str(cfg["weight_decay"]),\n        "--workers", str(protocol["workers"]),\n        "--label-smoothing", str(protocol["label_smoothing"]),\n        "--class-weight-alpha", str(protocol["class_weight_alpha"]),\n        "--ema-decay", str(protocol["ema_decay"]),\n        "--grad-clip", str(protocol["grad_clip"]),\n        "--warmup-fraction", str(protocol["warmup_fraction"]),\n        "--min-lr-factor", str(protocol.get("min_lr_factor", 0.01)),\n        "--rotation-deg", str(protocol["rotation_deg"]),\n        "--shuffle-mode", protocol["shuffle_mode"],\n        "--image-size", str(protocol["image_size"]),\n        "--seed", str(seed),\n    ]\n    if mode == "ddp":\n        # torchrun\'s first positional argument must be the training script.\n        return [sys.executable, "-m", "torch.distributed.run", "--standalone",\n                "--nnodes=1", "--nproc_per_node=2", str(script)] + args\n    # Single process on whichever GPU CUDA_VISIBLE_DEVICES exposes as 0.\n    return [sys.executable, str(script)] + args + ["--no-dist", "--device", device]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__,\n                                     formatter_class=argparse.RawDescriptionHelpFormatter)\n    parser.add_argument("--arch", required=True)\n    parser.add_argument("--seeds", nargs="+", type=int, required=True)\n    parser.add_argument("--mode", choices=("ddp", "single"), required=True)\n    parser.add_argument("--protocol", type=Path, required=True)\n    parser.add_argument("--train-images", type=Path, required=True)\n    parser.add_argument("--train-labels", type=Path, required=True)\n    parser.add_argument("--val-images", type=Path, required=True)\n    parser.add_argument("--val-labels", type=Path, required=True)\n    parser.add_argument("--classes-json", type=Path, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--engine", type=Path, default=ENGINE)\n    parser.add_argument("--device", default="cuda", choices=("auto", "cuda", "cpu"),\n                        help="single-gpu mode only; cpu exists for local tests")\n    parser.add_argument("--epochs", type=int, default=None,\n                        help="override protocol epochs; for fast local checks")\n    parser.add_argument("--verbose", action="store_true",\n                        help="echo each engine command before running it")\n    args = parser.parse_args()\n\n    protocol = json.loads(args.protocol.read_text(encoding="utf-8"))\n    if args.arch not in protocol["architectures"]:\n        raise SystemExit(f"{args.arch} is not in protocol.json")\n    cfg = protocol["architectures"][args.arch]\n    paths = {\n        "train_x": args.train_images, "train_y": args.train_labels,\n        "val_x": args.val_images, "val_y": args.val_labels,\n        "classes": args.classes_json,\n    }\n\n    print(f"[{args.arch}] mode={args.mode} seeds={args.seeds} "\n          f"gpus={os.environ.get(\'CUDA_VISIBLE_DEVICES\', \'all\')}", flush=True)\n\n    failures = []\n    for seed in args.seeds:\n        out_dir = args.output_root / args.arch / f"seed{seed}"\n        if (out_dir / "validation_metrics.json").is_file():\n            print(f"[{args.arch} seed {seed}] already complete, skipping", flush=True)\n            continue\n        out_dir.mkdir(parents=True, exist_ok=True)\n        epochs = args.epochs if args.epochs is not None else protocol["epochs"]\n        command = build_command(args.arch, seed, cfg, protocol, paths, out_dir,\n                                args.mode, args.engine, args.device, epochs)\n        if args.verbose:\n            print("  " + " ".join(command), flush=True)\n        started = time.time()\n        result = subprocess.run(command, env={**os.environ, "PYTHONUNBUFFERED": "1"}, check=False)\n        minutes = (time.time() - started) / 60.0\n        status = "ok" if result.returncode == 0 else f"FAILED ({result.returncode})"\n        print(f"[{args.arch} seed {seed}] {status} in {minutes:.2f} min", flush=True)\n        if result.returncode != 0:\n            failures.append(seed)\n            # Echo the command so a failure is diagnosable from the log alone.\n            print("  command: " + " ".join(command), flush=True)\n\n    if failures:\n        raise SystemExit(f"[{args.arch}] seeds failed: {failures}")\n    print(f"[{args.arch}] all requested seeds complete", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    '/kaggle/working/protocol_v2.json': '{\n  "version": 2,\n  "supersedes": "configs/protocol.json (v1). v1 was measured to be invalid on 2026-10-25: the backbone never fine-tuned, EfficientNet-B0 reached 0.6815 validation accuracy against 0.9954 for the reference. Evidence: reports/model_comparison/SWEEP_V1_INVALID.md",\n  "note": "Restores the optimisation settings that actually produced the reference checkpoints, instead of a modern recipe that was never validated. Every value below is traceable to a source: efficientnet_b0 to notebooks/01_Train_EfficientNet_B0.ipynb (the script that produced the 99.54% checkpoint), smallcnn to reports/phase2_baseline/experiment_config.json (96.01%). resnet18 and convnext_tiny have no recorded original config, so they inherit the validated efficientnet_b0 recipe.",\n  "image_size": 224,\n  "epochs": 12,\n  "patience": 3,\n  "seeds": [\n    42,\n    43,\n    44\n  ],\n  "eval_batch_size": 128,\n  "workers": 4,\n  "shuffle_mode": "random",\n  "rotation_deg": 12.0,\n  "grad_clip": 5.0,\n  "warmup_fraction": 0.0,\n  "min_lr_factor": 1.0,\n  "label_smoothing": 0.0,\n  "class_weight_alpha": 0.0,\n  "ema_decay": 0.0,\n  "selection_metric": "val_macro_f1",\n  "architectures": {\n    "smallcnn": {\n      "pretrained": false,\n      "local_batch_size": 32,\n      "lr": 0.001,\n      "weight_decay": 0.0001,\n      "why": "trains from scratch; values copied from reports/phase2_baseline/experiment_config.json (global batch 64 = 32 per GPU, AdamW lr 1e-3, weight decay 1e-4), which produced 96.01% validation accuracy. This is the only sanctioned deviation from the shared recipe."\n    },\n    "resnet18": {\n      "pretrained": true,\n      "local_batch_size": 16,\n      "lr": 0.0001,\n      "weight_decay": 0.0001,\n      "why": "no original config is on record for the 96.15% ResNet18 checkpoint, so it inherits the validated efficientnet_b0 recipe unchanged rather than a guessed one."\n    },\n    "efficientnet_b0": {\n      "pretrained": true,\n      "local_batch_size": 16,\n      "lr": 0.0001,\n      "weight_decay": 0.0001,\n      "why": "copied from notebooks/01_Train_EfficientNet_B0.ipynb: AdamW lr 1e-4, weight decay 1e-4, global batch 32 (16 per GPU), constant learning rate, no label smoothing, no class weights. This produced the 99.54% reference checkpoint, reaching 98.26% validation accuracy in epoch 1."\n    },\n    "convnext_tiny": {\n      "pretrained": true,\n      "local_batch_size": 16,\n      "lr": 0.0001,\n      "weight_decay": 0.0001,\n      "why": "no original config on record; inherits the validated efficientnet_b0 recipe. Its 2018-era-era activation cost is higher, but at 16 per GPU it fits, and keeping the recipe identical is what makes the cross-architecture check meaningful."\n    }\n  },\n  "notes": {\n    "constant_lr": "warmup_fraction 0.0 with min_lr_factor 1.0 makes the schedule flat: lr_lambda returns 1.0 for every step. This reproduces the constant learning rate of the original run. Setting either value back to its v1 value re-introduces decay.",\n    "what_v1_got_wrong": "v1 used weight decay 0.05 (500x the proven 1e-4), global batch 128 (4x fewer optimizer steps than the proven 32), warmup plus cosine decay to near zero (the proven run held the rate constant), and label smoothing 0.05 with class weights (the proven run used neither).",\n    "class_balancing": "Class-weighted loss and label smoothing are NOT part of the baseline arm. They belong to the H3 treatment arm and are introduced there, separately, so that if they change the result we know it was them.",\n    "augmentation": "Horizontal flip, rotation up to 12 degrees, brightness 0.88-1.12 and per-channel gain 0.92-1.08, applied on the GPU inside the engine. The original run used flip, rotation and brightness; the per-channel gain is the only addition, and it is kept because intensity variation is one of the documented differences between PlantVillage and field images.",\n    "gate": "No sweep may run until one 2-epoch EfficientNet-B0 run reaches 0.95 validation accuracy by epoch 2. The original reached 0.9826 in epoch 1, so the gate is deliberately loose: it detects a broken configuration, not a slightly weaker one.",\n    "shuffle": "v2 initially inherited shuffle_mode \'blocked\' from v1. Measured on the real manifest (scripts/check_batch_purity.py): the manifest is written in class order, so permuting 2048-image blocks and shuffling inside them yields batches with a median of 3 distinct classes, against 12 under a full random shuffle. That is a BatchNorm problem: a batch drawn from three classes produces class-specific normalisation statistics. It also explains the v1 sweep pattern, where the only architecture that trained (ConvNeXt, 0.95) uses LayerNorm and the three that collapsed (EfficientNet-B0 0.68, ResNet18 0.58, SmallCNN 0.23) all use BatchNorm. Random shuffle removes the effect. The original run that reached 0.9826 in one epoch used an ordinary shuffled DataLoader."\n  }\n}\n',
}

for _path, _content in FILES.items():
    _target = Path(_path)
    _target.parent.mkdir(parents=True, exist_ok=True)
    _target.write_text(_content, encoding='utf-8')
    print(f'wrote {_target.name:20s} {len(_content):7d} bytes')

import py_compile
for _file in (SCRIPT, WORKER):
    py_compile.compile(str(_file), doraise=True)
print('engine and worker compile')

PROTOCOL = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
assert PROTOCOL['version'] == 2, PROTOCOL['version']
assert PROTOCOL['shuffle_mode'] == 'random', ('blocked shuffling breaks BatchNorm: see R14', PROTOCOL['shuffle_mode'])
assert PROTOCOL['warmup_fraction'] == 0.0 and PROTOCOL['min_lr_factor'] == 1.0, 'v2 must hold the LR constant'
assert PROTOCOL['label_smoothing'] == 0.0 and PROTOCOL['class_weight_alpha'] == 0.0, 'baseline arm uses no smoothing or class weights'
b0 = PROTOCOL['architectures']['efficientnet_b0']
assert b0['lr'] == 1e-4 and b0['weight_decay'] == 1e-4 and b0['local_batch_size'] == 16, b0
print(f"protocol v{PROTOCOL['version']} | shuffle={PROTOCOL['shuffle_mode']} | flat LR | "
      f"b0 lr={b0['lr']} wd={b0['weight_decay']} batch/gpu={b0['local_batch_size']}")
print('protocol assertions passed')

## 4. Gate

In [ ]:
# Measured on 2x T4 during the pilot; only used to project the schedule.
PILOT_MINUTES = {"smallcnn": 0.67, "resnet18": 0.78, "efficientnet_b0": 2.45, "convnext_tiny": 3.15}

if GATE_ROOT.exists():
    shutil.rmtree(GATE_ROOT)                 # the gate always runs fresh

print("=" * 70)
print("GATE — EfficientNet-B0, seed 42, 2 epochs")
print("=" * 70, flush=True)
gate_started = time.time()
gate = subprocess.run(worker_command("efficientnet_b0", "ddp", [42], GATE_EPOCHS, GATE_ROOT),
                      env={**os.environ, "PYTHONUNBUFFERED": "1"}, check=False)
gate_minutes = (time.time() - gate_started) / 60.0

GATE_PASSED = False
gate_history = None
if gate.returncode != 0:
    print(f"\nGATE ERROR: the gate run crashed (exit {gate.returncode}).")
    log({"stage": "gate", "status": "crashed", "exit": gate.returncode,
         "minutes": round(gate_minutes, 2)})
else:
    gate_history = pd.read_csv(GATE_ROOT / "efficientnet_b0" / "seed42" / "training_history.csv")
    print("\n" + gate_history[["epoch", "train_loss", "val_accuracy", "lr"]].to_string(index=False))
    final_accuracy = float(gate_history["val_accuracy"].iloc[-1])
    first_accuracy = float(gate_history["val_accuracy"].iloc[0])
    MINUTES_PER_EPOCH = gate_minutes / GATE_EPOCHS
    GATE_PASSED = final_accuracy >= GATE_THRESHOLD

    print(f"\nepoch 1 accuracy : {first_accuracy:.4f}  (original recipe: 0.9826)")
    print(f"epoch 2 accuracy : {final_accuracy:.4f}  (threshold: {GATE_THRESHOLD})")
    print(f"throughput       : {MINUTES_PER_EPOCH:.2f} min/epoch")
    log({"stage": "gate", "status": "passed" if GATE_PASSED else "failed",
         "epoch1": round(first_accuracy, 4), "epoch2": round(final_accuracy, 4),
         "minutes_per_epoch": round(MINUTES_PER_EPOCH, 3), "minutes": round(gate_minutes, 2)})

if not GATE_PASSED:
    print("\n" + "!" * 70)
    print("THE SWEEP IS CANCELLED. Nothing long will run tonight.")
    print("The gate did its job: it cost ~6 minutes instead of ~6 hours.")
    print("The results cell below still runs and still writes a zip containing")
    print("the gate history — that is the evidence needed to diagnose this.")
    print("!" * 70)
else:
    print(f"\nGATE PASSED. Full sweep released.")

## 4. The sweep

Runs in priority order: the reference model first, so if anything is cut short the
most important numbers exist. Each run is a separate subprocess with a timeout.
A failed or timed-out run is recorded and the loop moves on — one bad seed never
costs the night.

Finished seeds are skipped, so re-running this notebook after a crash resumes
rather than repeating work.

The last stage adds two extra EfficientNet-B0 seeds, but only if the remaining
budget clearly covers them.

In [ ]:
if not GATE_PASSED:
    print("Skipped: the gate did not pass.")
else:
    PROTOCOL = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
    SEEDS = PROTOCOL["seeds"]
    EPOCHS = PROTOCOL["epochs"]
    SCALE = {a: PILOT_MINUTES[a] / PILOT_MINUTES["efficientnet_b0"] for a in PILOT_MINUTES}

    # Reference model first: it carries H1-H4, so it must survive any cut-off.
    PLAN = [
        ("efficientnet_b0", SEEDS),
        ("convnext_tiny", SEEDS),
        ("resnet18", SEEDS),
        ("smallcnn", SEEDS),
    ]
    if ALLOW_EXTRA_SEEDS:
        PLAN.append(("efficientnet_b0", [45, 46]))

    def remaining_hours():
        return SESSION_BUDGET_H - (time.time() - STARTED_AT) / 3600.0

    def already_done(arch, seed):
        return (OUTPUT_ROOT / arch / f"seed{seed}" / "validation_metrics.json").is_file()

    failures, completed, skipped = [], [], []

    for arch, seeds in PLAN:
        for seed in seeds:
            if already_done(arch, seed):
                skipped.append(f"{arch}/seed{seed}")
                print(f"[skip] {arch} seed {seed} — already complete")
                continue

            estimate = EPOCHS * MINUTES_PER_EPOCH * SCALE[arch]
            budget_left = remaining_hours() - RESERVE_H
            if budget_left * 60 < estimate * 1.25:
                print(f"[cut] {arch} seed {seed} — needs ~{estimate:.0f} min, "
                      f"only {budget_left*60:.0f} min of budget left")
                failures.append({"arch": arch, "seed": seed, "reason": "budget"})
                continue

            timeout = max(1800, int(estimate * 60 * 4))
            print(f"\n{'='*70}\n[{arch} seed {seed}] start {time.strftime('%H:%M')} | "
                  f"estimate {estimate:.0f} min | timeout {timeout//60} min | "
                  f"budget left {budget_left:.1f} h\n{'='*70}", flush=True)
            run_started = time.time()
            try:
                result = subprocess.run(worker_command(arch, "ddp", [seed], EPOCHS, OUTPUT_ROOT),
                                        env={**os.environ, "PYTHONUNBUFFERED": "1"},
                                        check=False, timeout=timeout)
                elapsed = (time.time() - run_started) / 60.0
                if result.returncode == 0:
                    completed.append(f"{arch}/seed{seed}")
                    print(f"[done] {arch} seed {seed} in {elapsed:.1f} min")
                    log({"stage": "run", "arch": arch, "seed": seed, "status": "ok",
                         "minutes": round(elapsed, 2)})
                else:
                    failures.append({"arch": arch, "seed": seed, "reason": f"exit {result.returncode}"})
                    print(f"[FAIL] {arch} seed {seed} exit {result.returncode} after {elapsed:.1f} min")
                    log({"stage": "run", "arch": arch, "seed": seed, "status": "failed",
                         "exit": result.returncode, "minutes": round(elapsed, 2)})
            except subprocess.TimeoutExpired:
                elapsed = (time.time() - run_started) / 60.0
                failures.append({"arch": arch, "seed": seed, "reason": "timeout"})
                print(f"[TIMEOUT] {arch} seed {seed} killed after {elapsed:.1f} min")
                log({"stage": "run", "arch": arch, "seed": seed, "status": "timeout",
                     "minutes": round(elapsed, 2)})
            except Exception as exc:
                failures.append({"arch": arch, "seed": seed, "reason": repr(exc)})
                print(f"[ERROR] {arch} seed {seed}: {exc!r}")
                log({"stage": "run", "arch": arch, "seed": seed, "status": "error",
                     "error": repr(exc)})

    print("\n" + "=" * 70)
    print(f"completed {len(completed)} | skipped {len(skipped)} | failed {len(failures)}")
    if failures:
        for item in failures:
            print(f"  failed: {item}")
    print(f"session time so far: {(time.time() - STARTED_AT)/3600:.2f} h")
    print("=" * 70)

## 5. Results and bundle

In [ ]:
records = []
for metrics_path in sorted(OUTPUT_ROOT.rglob("validation_metrics.json")):
    parts = metrics_path.relative_to(OUTPUT_ROOT).parts
    if any(skip in parts for skip in ("_smoke", "pilot", "_gate")):
        continue
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    records.append({
        "architecture": metrics.get("model"),
        "seed": int(metrics_path.parent.name.replace("seed", "") or 0),
        "val_accuracy": metrics.get("val_accuracy"),
        "val_macro_f1": metrics.get("val_macro_f1"),
        "val_ece": metrics.get("val_ece"),
        "best_epoch": metrics.get("best_epoch"),
        "train_minutes": metrics.get("train_minutes"),
        "global_batch": metrics.get("global_batch_size"),
        "weight_decay": metrics.get("weight_decay"),
        "fingerprint": (metrics.get("dataset_fingerprint") or "")[:12],
    })

zip_name = "/kaggle/working/overnight_runs"
if records:
    comparison = pd.DataFrame(records).sort_values(["architecture", "seed"]).reset_index(drop=True)
    comparison.to_csv(OUTPUT_ROOT / "comparison.csv", index=False)
    display(comparison)

    print("\nfingerprints (must agree within an architecture):")
    print(comparison.groupby("architecture")["fingerprint"].apply(lambda c: sorted(set(c))).to_string())

    summary = (comparison.groupby("architecture")["val_macro_f1"]
               .agg(runs="count", mean="mean", std="std", min="min", max="max").round(6)
               .sort_values("mean", ascending=False))
    summary["spread_pp"] = ((summary["max"] - summary["min"]) * 100).round(2)
    print("\nValidation macro-F1 per architecture")
    display(summary)

    reference = comparison[comparison.architecture == "efficientnet_b0"]
    if not reference.empty:
        print(f"\nReference model: {len(reference)} seeds, macro-F1 "
              f"{reference.val_macro_f1.mean():.4f} +/- {reference.val_macro_f1.std():.4f} "
              f"(published checkpoint: 0.99369)")

    if summary["runs"].max() > 1:
        fig, ax = plt.subplots(figsize=(9, 4))
        order = list(summary.index)
        for i, arch in enumerate(order):
            values = comparison.loc[comparison.architecture == arch, "val_macro_f1"]
            ax.scatter([i] * len(values), values, s=70, zorder=3)
            ax.hlines(values.mean(), i - 0.22, i + 0.22, colors="black", linewidth=2, zorder=4)
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels(order, rotation=12)
        ax.set_ylabel("Validation macro-F1")
        ax.set_title("Per-architecture validation macro-F1 (points = seeds, bar = mean)")
        ax.grid(axis="y", alpha=0.3)
        fig.tight_layout()
        fig.savefig(OUTPUT_ROOT / "macro_f1_by_architecture.png", dpi=160)
        plt.show()
        plt.close(fig)
else:
    print("No completed runs to report.")
    print("If the gate failed, that is expected: the zip will still contain the gate evidence.")

# Always ship something, even if every run failed.
zip_path = shutil.make_archive(zip_name, "zip", root_dir=OUTPUT_ROOT)
print(f"\nBundle: {zip_path}  ({Path(zip_path).stat().st_size/1e6:.0f} MB)")
print(f"Session total: {(time.time() - STARTED_AT)/3600:.2f} h")
print("Download it from the Output panel, or fetch it later with:")
print("  kaggle kernels output <user>/<notebook> -p ./out")

## In the morning

1. Download `overnight_runs.zip`.
2. Check the summary printed above against the published reference: the
   EfficientNet-B0 mean macro-F1 should land near **0.9937** at 12 epochs. If it
   is far below that, stop — do not build the comparison table on it.
3. Check `_overnight_log.json` inside the zip for any `failed` or `timeout`
   entries.
4. Read the seed spread. A difference between architectures that is smaller than
   the spread within an architecture is not a result, and must not be written up
   as one.

Nothing here may be quoted in a document until it is registered in
`results/registry.json` with its checkpoint hash and fingerprint.